# Stage 3 dataset validation

This notebook runs the same reusable checks used by the command-line validator and makes the main coverage and safety results easy to inspect.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'PROJECT_PLAN.md').exists():
    project_root = project_root.parent
if not (project_root / 'PROJECT_PLAN.md').exists():
    raise RuntimeError('Run this notebook from the repository or its notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset_io import read_jsonl, read_source_catalog
from src.dataset_validator import validate_dataset

processed_dir = project_root / 'data' / 'processed'
corpus = read_jsonl(processed_dir / 'knowledge_corpus.jsonl')
queries = read_jsonl(processed_dir / 'retrieval_evaluation_queries.jsonl')
report = validate_dataset(project_root)
print('Validation passed:', report['passed'])
print('Errors:', report['error_count'], 'Warnings:', report['warning_count'])

## Counts and review status

In [ ]:
def show_counter(label, values):
    print(f'\n{label}')
    for key, count in sorted(Counter(values).items(), key=lambda item: str(item[0])):
        print(f'  {key}: {count}')

show_counter('Document family', (record['document_family'] for record in corpus))
show_counter('Subject family', (record.get('subject_family', 'not_applicable') for record in corpus))
show_counter('Educational level', (level for record in corpus for level in record.get('educational_level', [])))
show_counter('Canonical versus synthetic', ('synthetic' if record['synthetic'] else 'canonical' for record in corpus))
show_counter('Review status', ('reviewed' if record['reviewed'] else 'unreviewed_tier_c' for record in corpus))

## Sources, duplicates, and coverage

In [ ]:
catalog = read_source_catalog(project_root / 'data' / 'sources' / 'source_catalog.csv')
source_usage = Counter(source_id for record in corpus for source_id in record['source_ids'])
print('Catalog sources:', len(catalog), 'Used sources:', len(source_usage))
print('Most-used sources:')
for source_id, count in source_usage.most_common(10):
    print(f'  {source_id}: {count}')
near = report['checks']['near_duplicate_retrieval_texts']
print('Exact duplicate groups:', len(report['checks']['duplicate_records_and_retrieval_texts']['groups']))
print('Near-duplicate review candidates:', near['candidate_count'])
print('Missing subject families:', report['checks']['subject_family_coverage']['missing'])
print('Missing education levels:', report['checks']['educational_level_coverage']['missing'])

## Sensor safety and evaluation separation

In [ ]:
sensor_check = report['checks']['sensor_safety_and_medical_language']
separation = report['checks']['evaluation_corpus_separation']
print('Sensor safety passed:', sensor_check['passed'])
print('Invalid sensor records:', sensor_check['invalid_records'])
print('Evaluation queries:', len(queries))
print('Evaluation leakage:', separation['leaked_queries'])
print('Evaluation records inside corpus:', sum(record['document_family'] == 'retrieval_evaluation_query' for record in corpus))

## Conclusion

In [ ]:
if report['passed']:
    print('The Stage 3 corpus passes all blocking checks. Any listed near-duplicate candidates remain explicit review items, not hidden failures.')
else:
    print('The corpus is not ready. Fix these failures before Stage 4:')
    for error in report['errors']:
        print('-', error)